# 2.2 Randomized Measurement

## WaveFunction Overlap

```{seealso}

You can download the Jupyter Notebook for this section from the link {nb-download}`./pp_2_2_overlap_randomized.ipynb`.
```

This method is based on [Probing Rényi entanglement entropy via randomized measurements](https://www.science.org/doi/abs/10.1126/science.aau4963) with deplorizing error mitigation by [Simple mitigation of global depolarizing errors in quantum simulations](https://link.aps.org/doi/10.1103/PhysRevE.104.035309).


## `randomized_overlap_echo`


Here, we assume you already have the calculated data. You can call the function `randomized_overlap_echo` from `qurry.process.randomized_measure` to calculate the entropy.

This function requires only three arguments: `shots`, `counts`, and `selected_classical_registers`. All other arguments are optional.

Note that `selected_classical_registers` expects a list of <u>bitstring indices</u>, **NOT** <u>qubit indices</u>, as these are distinct.

The following is the arguments of the function:

```python

def randomized_overlap_echo(
    shots: int,
    first_counts: list[dict[str, int]],
    second_counts: list[dict[str, int]],
    selected_classical_registers: Optional[Iterable[int]] = None,
    backend: PostProcessingBackendLabel = DEFAULT_PROCESS_BACKEND,
    pbar: Optional[tqdm.tqdm] = None,
) -> WaveFuctionOverlapResult:
    """Calculate wavefunction overlap
    a.k.a. loschmidt echo when processes time evolution system.

    Args:
        shots (int):
            Shots of the experiment on quantum machine.
        first_counts (list[dict[str, int]]):
            Counts of the experiment on quantum machine.
        second_counts (list[dict[str, int]]):
            Counts of the experiment on quantum machine.
        selected_classical_registers (Optional[Iterable[int]], optional):
            The list of **the index of the selected_classical_registers**.
        backend (ExistingProcessBackendLabel, optional):
            Backend for the process. Defaults to DEFAULT_PROCESS_BACKEND.
        pbar (Optional[tqdm.tqdm], optional):
            The progress bar API, you can use put a :cls:`tqdm` object here.
            This function will update the progress bar description.
            Defaults to None.


    Returns:
        WaveFuctionOverlapResult: A dictionary contains purity, entropy,
            a list of each overlap, puritySD, degree, actual measure range, bitstring range.
    """
```

This function returns a dictionary that contains the echo, echoSD, and other relevant information.
The returned `dict` contains the following information:

```python
class WaveFunctionOverlapResult(TypedDict):
    """The return type of the post-processing for wavefunction overlap."""

    echo: FloatType
    """The overlap value."""
    echo_sd: FloatType
    """The overlap standard deviation."""
    echo_cells: dict[int, np.float64] | dict[int, float]
    """The overlap of each single count."""
    num_classical_registers: int
    """The number of classical registers."""
    classical_registers: list[int] | None
    """The list of the index of the selected classical registers."""
    classical_registers_actually: list[int]
    """The list of the index of the selected classical registers which is actually used."""
    # refactored
    taking_time: float
    """The calculation time."""
    counts_num: int
    """The number of first counts and second counts."""
```


### Dummy Data


```{warning}

The dummy data is just some random bitstrings for the demonstration of data structure.
They have no physical meaning.
```

```{eval-rst}

.. seealso::

   You can download the dummy data for this section from the link :download:`./easy-dummy.json`.
```


In [2]:
import json

with open("./easy-dummy.json", "r") as f:
    easy_dummy = json.load(f)
large_dummy_list = [easy_dummy["0"] for _ in range(100)]

### Simple Example


In [3]:
from qurry.process.randomized_measure import randomized_overlap_echo

test_result_1 = randomized_overlap_echo(
    4096, large_dummy_list, large_dummy_list, list(range(6))
)

In [4]:
from pprint import pprint

print("| result of 'randomized_overlap_echo' except for 'echo_cells'")
pprint({k: v for k, v in test_result_1.items() if k != "echo_cells"})

print("-" * 50)
print("| Length of 'echo_cells':", len(test_result_1["echo_cells"]))
print("| Some result of 'echo_cells'")
pprint(dict(list(test_result_1["echo_cells"].items())[:10]))


| result of 'randomized_overlap_echo' except for 'echo_cells'
{'classical_registers': [0, 1, 2, 3, 4, 5],
 'classical_registers_actually': [0, 1, 2, 3, 4, 5],
 'counts_num': 100,
 'echo': np.float64(1.0627930164337158),
 'echo_sd': np.float64(0.0),
 'num_classical_registers': 8,
 'taking_time': 0.003610682}
--------------------------------------------------
| Length of 'echo_cells': 100
| Some result of 'echo_cells'
{4: 1.0627930164337158,
 12: 1.0627930164337158,
 25: 1.0627930164337158,
 26: 1.0627930164337158,
 34: 1.0627930164337158,
 46: 1.0627930164337158,
 60: 1.0627930164337158,
 75: 1.0627930164337158,
 90: 1.0627930164337158,
 92: 1.0627930164337158}


### Integration with your own progress bar


In [ ]:
from tqdm import tqdm
import time

all_counts_progress_01 = tqdm(
    [
        (4096, large_dummy_list, list(range(6))),
        (4096, large_dummy_list, list(range(2, 8))),
        (4096, large_dummy_list, list(range(7))),
        (4096, large_dummy_list, list(range(0, 7))),
        (4096, large_dummy_list, [0, 1, 2, 6, 7]),
        (4096, large_dummy_list, [3, 4, 5, 6, 7, 0, 1]),
    ],
    bar_format="| {desc} - {elapsed} < {remaining}",
)

test_result_2 = []
for tmp_shot, tmp_counts, tmp_partition in all_counts_progress_01:
    print(tmp_partition)
    test_result_2.append(
        randomized_overlap_echo(
            tmp_shot, tmp_counts, tmp_counts, tmp_partition, pbar=all_counts_progress_01
        )
    )
    time.sleep(0.5)

| Calculate selected classical registers: [0, 1, 2, 3, 4, 5]. - 00:00 < ?

[0, 1, 2, 3, 4, 5]


| Calculate selected classical registers: [2, 3, 4, 5, 6, 7]. - 00:00 < 00:02

[2, 3, 4, 5, 6, 7]


| Calculate selected classical registers: [0, 1, 2, 3, 4, 5, 6]. - 00:01 < 00:02

...

| Calculate selected classical registers: [0, 1, 2, 6, 7]. - 00:02 < 00:01      

[0, 1, 2, 6, 7]


| Calculate selected classical registers: [3, 4, 5, 6, 7, 0, 1]. - 00:02 < 00:00

[3, 4, 5, 6, 7, 0, 1]


| Calculate selected classical registers: [3, 4, 5, 6, 7, 0, 1]. - 00:03 < 00:00


### Using Python backend

It will be slow. You'd better think twice before using it.


In [6]:
all_counts_progress_02 = tqdm(
    [
        (4096, large_dummy_list, list(range(6))),
        (4096, large_dummy_list, list(range(2, 8))),
        (4096, large_dummy_list, list(range(7))),
        (4096, large_dummy_list, list(range(0, 7))),
        (4096, large_dummy_list, [0, 1, 2, 6, 7]),
        (4096, large_dummy_list, [3, 4, 5, 6, 7, 0, 1]),
    ],
    bar_format="| {desc} - {elapsed} < {remaining}",
)


test_result_2 = []
for tmp_shot, tmp_counts, tmp_partition in all_counts_progress_02:
    print(tmp_partition)
    test_result_2.append(
        randomized_overlap_echo(
            tmp_shot,
            tmp_counts,
            tmp_counts,
            tmp_partition,
            pbar=all_counts_progress_02,
            backend="Python",
        )
    )
    time.sleep(0.5)

| Calculate selected classical registers: [0, 1, 2, 3, 4, 5]. - 00:00 < ?

[0, 1, 2, 3, 4, 5]


| Calculate selected classical registers: [2, 3, 4, 5, 6, 7]. - 00:02 < 00:10

[2, 3, 4, 5, 6, 7]


| Calculate selected classical registers: [0, 1, 2, 3, 4, 5, 6]. - 00:04 < 00:08

...

| Calculate selected classical registers: [0, 1, 2, 6, 7]. - 00:08 < 00:04      

[0, 1, 2, 6, 7]


| Calculate selected classical registers: [3, 4, 5, 6, 7, 0, 1]. - 00:10 < 00:02

[3, 4, 5, 6, 7, 0, 1]


| Calculate selected classical registers: [3, 4, 5, 6, 7, 0, 1]. - 00:13 < 00:00


## Post-Process Availablities and Version Info


In [7]:
from qurry.process.status import AVAIBILITY_STATESHEET

AVAIBILITY_STATESHEET

 | Qurrium version: 1.0.0.dev2
---------------------------------------------------------------------------
 ### Qurrium Post-Processing
   - Backend Availability ................... Python Cython Rust   JAX   
 - randomized_measure
   - entangled_entropy.entropy_core_2 ....... Yes    No     Yes    No    
   - entangled_entropy_v1.entropy_core ...... Yes    Depr.  Yes    No    
   - wavefunction_overlap.echo_core_2 ....... Yes    No     Yes    No    
   - wavefunction_overlap_v1.echo_core ...... Yes    Depr.  Yes    No    
 - hadamard_test
   - purity_echo_core ....................... Yes    No     Yes    No    
 - magnet_square
   - magnsq_core ............................ Yes    No     Yes    No    
 - string_operator
   - strop_core ............................. Yes    No     Yes    No    
 - classical_shadow
   - rho_process ............................ Yes    No     Yes    No    
   - matrix_calculation ..................... Yes    No     No     Yes   
 - utils
   - randomized ....